# Multi-horizon XGBoost — A-share trend prediction

Trains one **XGBoost** multiclass classifier **per horizon (1/5/20 days)** on tabular features projected from the same windowed split the TCN uses, each predicting one of **7 return buckets** (down >5% ... flat ... up >5%; edges scale with √h).

**How to run via the VS Code Google Colab extension:**
1. Open this notebook in VS Code.
2. Top-right kernel picker → connect to a **Colab** runtime (pick a **GPU** runtime for speed).
3. Run the cells top to bottom.

The Colab runtime is a fresh cloud VM, so it does **not** see your local repo. Set `REPO_URL` below to your pushed GitHub repo (or upload the `src/` folder). If you run this notebook on a *local* kernel instead, it imports `src` directly.

In [ ]:
import os, sys, subprocess, shutil

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
REPO_URL = "https://github.com/LiangS/Peng.git"  # set to your repo; blank if src/ already present
REPO_DIR = "/content/Peng"

if IN_COLAB and REPO_URL:
    os.chdir("/content")                                  # never operate from inside a stale checkout
    nested = os.path.join(REPO_DIR, "Peng")
    if os.path.exists(nested):
        shutil.rmtree(nested)                             # clean up an old nested clone
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)  # always get latest
    os.chdir(REPO_DIR)
elif not os.path.exists("src") and os.path.exists("../src"):
    os.chdir("..")  # running from notebooks/ on a local kernel

print("cwd:", os.getcwd(), "| in_colab:", IN_COLAB)
assert os.path.exists("src"), "src/ not found — set REPO_URL or upload the package"

In [ ]:
!pip -q install -r requirements.txt
import xgboost as xgb
print("xgboost", xgb.__version__)

## Train per-horizon XGBoost

`train_xgb` builds the shared windowed split, projects it to tabular features (lags + mean/std), fits one class-weighted classifier per horizon, and prints accuracy / macro-F1 against the majority-class baseline.

> **Note:** XGBoost is CPU-bound here, so a GPU runtime is not required.

In [ ]:
from src.config import Config
from src.train_xgb import train_xgb

cfg = Config(
    symbol="600519",        # Kweichow Moutai; change to your A-share code
    horizons=[1, 5, 20],
    window=60,
)
report, models = train_xgb(cfg)

## Visualize training

Plot each per-horizon classifier's validation log-loss across boosting rounds (from `evals_result()`).

In [ ]:
from src.plotting import plot_xgb_evals

fig = plot_xgb_evals(models, cfg.horizons)

## Inference — per-horizon direction probabilities

Load the saved per-horizon models and print class probabilities for the most recent window.

In [ ]:
import os
import numpy as np
import xgboost as xgb
from src.data import load_prices
from src.features import build_dataset_frame
from src.prepare import prepare_windowed
from src.tabular import window_to_tabular

# train_xgb doesn't persist the scaler, so rebuild the (deterministic, train-only)
# scaler from the same split the models were trained on, then scale the last window
# the SAME way before projecting to tabular features.
data = prepare_windowed(cfg)

prices = load_prices(cfg.symbol, cfg.start_date, cfg.end_date, cfg.adjust, cfg.cache_dir)
feats, _ = build_dataset_frame(prices, cfg.horizons, cfg.class_thresholds)

window = cfg.window
raw = feats.to_numpy()[-window:]                          # (window, F), order = FEATURE_COLUMNS
scaled = (raw - data.scaler_mean) / data.scaler_scale
last_window = scaled.T[None]                              # (1, F, window)
X_last, _ = window_to_tabular(last_window, data.feature_names)

outdir = os.path.join(cfg.ckpt_dir, f"xgb_{cfg.symbol}")
from src.features import class_names
classes = class_names(cfg.class_thresholds)
print(f"As of {feats.index[-1].date()} ({cfg.symbol}):\n")
for h in cfg.horizons:
    clf = xgb.XGBClassifier()
    clf.load_model(os.path.join(outdir, f"h{h}.json"))
    p = clf.predict_proba(X_last)[0]
    pretty = "  ".join(f"{c}={v:.2f}" for c, v in zip(classes, p))
    print(f"  {h:>2}d  ->  {pretty}   (argmax: {classes[int(np.argmax(p))]})")